# AgentNet (Ubuntu) screenshot exploratory data analysis

**Goal.** Describe the screenshots that accompany the Ubuntu trajectories: resolution, file
size, how they differ by domain, and whether the downloaded sample represents the whole
Ubuntu subset.

**Input.** A seeded, domain stratified sample of tasks, fetched by
`scripts/download_agentnet_images.py`. The revision, seed and requested task count are in
`data/raw/agentnet/ubuntu_images_manifest.csv`. Every image was CRC32 verified on download
and its sha256 is stored in the manifest.

**Conclusion.** The sample has 103 tasks, 1,716 steps and 1,715 unique images, with none missing on disk and no
sha256 mismatch. One image name is reused by two steps. Every image is 1920 by 1080 with a mean
uncompressed size of 0.92 MB. Domain shares and mean trajectory length match the full Ubuntu
subset closely (16.7 versus 16.5 steps), so the sample is representative on these checks. The scored
rate differs slightly (49.5% versus 53.5%).

In [1]:
import sys

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Python 3.14 is required. Run `uv python install 3.14` and `uv sync`, "
        "then open this notebook with `uv run jupyter lab notebooks/`."
    )

In [2]:
import hashlib
from io import BytesIO

import plotly.express as px
import polars as pl
from IPython.display import Image, Markdown, display
from ipywidgets import Dropdown, interactive_output
from PIL import Image as PilImage

from agentnet import (
    DATA_DIR,
    IMAGE_MANIFEST_FILE,
    IMAGES_DIR,
    SUBSET_FILES,
    png_size,
    scan_jsonl,
    task_table,
)

In [3]:
manifest = pl.read_csv(IMAGE_MANIFEST_FILE)
Markdown(
    f"Dataset revision `{manifest['revision'][0]}`, seed `{manifest['seed'][0]}`, "
    f"requested tasks `{manifest['requested_tasks'][0]}`."
)

Dataset revision `d76ee50a63fad81cfdbe576416757d7c2091ed50`, seed `2026`, requested tasks `100`.

## Integrity

In [4]:
unique = manifest.unique("name")
on_disk = [(IMAGES_DIR / name).exists() for name in unique["name"]]
mismatched = sum(
    hashlib.sha256((IMAGES_DIR / name).read_bytes()).hexdigest() != digest
    for name, digest in zip(unique["name"], unique["sha256"])
)
pl.DataFrame(
    {
        "tasks": [manifest["task_id"].n_unique()],
        "steps": [manifest.height],
        "unique_images": [unique.height],
        "missing_on_disk": [on_disk.count(False)],
        "sha256_mismatch": [mismatched],
        "reused_image_names": [manifest.height - unique.height],
    }
)

tasks,steps,unique_images,missing_on_disk,sha256_mismatch,reused_image_names
i64,i64,i64,i64,i64,i64
103,1716,1715,0,0,1


## Representativeness of the sample

The sample is stratified by domain, so domain shares should match the full Ubuntu subset.
Trajectory length is not a stratification variable, so it is a genuine check.

In [5]:
population = task_table(scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])).collect()
sampled_ids = manifest["task_id"].unique()
sample = population.filter(pl.col("task_id").is_in(sampled_ids.implode()))
comparison = (
    population.group_by("domain")
    .agg(pl.len().alias("population_tasks"), pl.col("n_steps").mean().alias("population_steps"))
    .join(
        sample.group_by("domain").agg(
            pl.len().alias("sample_tasks"), pl.col("n_steps").mean().alias("sample_steps")
        ),
        on="domain",
    )
    .with_columns(
        (pl.col("population_tasks") / population.height).alias("population_share"),
        (pl.col("sample_tasks") / sample.height).alias("sample_share"),
    )
    .sort("domain")
)
overall = pl.DataFrame(
    {
        "mean_steps_population": [population["n_steps"].mean()],
        "mean_steps_sample": [sample["n_steps"].mean()],
        "scored_rate_population": [population["task_completed"].is_not_null().mean()],
        "scored_rate_sample": [sample["task_completed"].is_not_null().mean()],
    }
)
display(overall)
comparison

mean_steps_population,mean_steps_sample,scored_rate_population,scored_rate_sample
f64,f64,f64,f64
16.4896,16.660194,0.5346,0.495146


domain,population_tasks,population_steps,sample_tasks,sample_steps,population_share,sample_share
str,u32,f64,u32,f64,f64,f64
"""Chrome""",718,15.442897,15,14.2,0.1436,0.145631
"""Gimp""",492,15.170732,10,12.0,0.0984,0.097087
"""MultiApp""",395,28.782278,8,30.0,0.079,0.07767
"""OS""",497,12.412475,10,12.6,0.0994,0.097087
"""Thunderbird""",300,13.64,6,12.333333,0.06,0.058252
…,…,…,…,…,…,…
"""error_correction""",200,18.52,4,21.25,0.04,0.038835
"""infeasible""",200,5.87,4,7.25,0.04,0.038835
"""libreoffice_calc""",298,19.761745,6,21.333333,0.0596,0.058252


In [6]:
sizes = unique.with_columns(
    pl.Series(
        "dimensions",
        [png_size(IMAGES_DIR / name) for name in unique["name"]],
        dtype=pl.Array(pl.Int64, 2),
    )
).with_columns(
    pl.col("dimensions").arr.get(0).alias("width"),
    pl.col("dimensions").arr.get(1).alias("height"),
    (pl.col("uncompressed_bytes") / 1e6).alias("megabytes"),
)

## Resolutions

In [7]:
sizes.group_by("width", "height").agg(
    pl.len().alias("images"), pl.col("megabytes").mean().alias("mean_megabytes")
).sort("images", descending=True)

width,height,images,mean_megabytes
i64,i64,u32,f64
1920,1080,1715,0.921263


In [8]:
px.histogram(
    sizes,
    x="megabytes",
    nbins=60,
    labels={"megabytes": "Uncompressed size (MB)"},
    title="PNG file size",
).update_yaxes(title_text="Images")

In [9]:
px.box(
    sizes.sort("domain"),
    x="megabytes",
    y="domain",
    labels={"megabytes": "Uncompressed size (MB)", "domain": "Domain"},
    title="PNG file size by domain",
)

## Example trajectory

Pick a task to view its first steps. Screenshots are downscaled to keep the notebook small.

In [ ]:
task_ids = manifest["task_id"].unique().sort().to_list()
task_picker = Dropdown(options=task_ids, value=task_ids[0], description="Task")
task_picker

Dropdown(description='Task', options=('054374a8-ffa2-4acf-9f1e-7c11f2b2d66a', '059e23d7-c3c0-4beb-9cd7-1c3bf48…

In [15]:
def show_screenshot(name: str) -> Image:
    """Return a downscaled JPEG preview of one screenshot."""
    buffer = BytesIO()
    with PilImage.open(IMAGES_DIR / name) as screenshot:
        screenshot.convert("RGB").resize(
            (480, round(480 * screenshot.height / screenshot.width))
        ).save(buffer, format="JPEG", quality=70)
    return Image(data=buffer.getvalue(), format="jpeg")


def show_trajectory(task_id: str) -> None:
    """Display the instruction and the first steps of one trajectory."""
    trajectory = (
        scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])
        .filter(pl.col("task_id") == task_id)
        .select("instruction", "traj")
        .collect()
        .row(0, named=True)
    )
    display(Markdown(f"**Instruction.** {trajectory['instruction']}"))
    for step in trajectory["traj"][:3]:
        display(
            Markdown(
                f"Step {step['index']}: {step['value']['action']}  \n" f"`{step['value']['code']}`"
            )
        )
        display(show_screenshot(step["image"]))


interactive_output(show_trajectory, {"task_id": task_picker})

Output()